In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")

print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Load the utilities data
df = pd.read_excel("Utilities.xlsx", sheet_name="Usage")

# Quick first look
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

Shape: (38, 26)

Columns:
['Year', 'Month', 'Gas Use', 'Elec Use', 'Solar Production', 'Elec Import', 'Elec Export', 'EV Charger', 'Net Use', 'Net Use (-EV)', 'Heating Degree Days', 'Gas Cost', 'Elec Import Cost', 'Export Income', 'Feedback Costs', 'Net Elec Cost', 'Fixed Costs', 'EV Charger.1', 'Net House Elec Cost', 'Net House Cost (Gas & Elec)', 'Overall Cost', 'Remarks', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25']

First 5 rows:


,Year,Month,Gas Use,Elec Use,Solar Production,Elec Import,Elec Export,EV Charger,Net Use,Net Use (-EV),...,Fixed Costs,EV Charger.1,Net House Elec Cost,Net House Cost (Gas & Elec),Overall Cost,Remarks,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,2023,Jan,0,0,0,0,0,0.0,0,0.0,...,0.00,0.0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN
1,2023,Feb,0,0,0,0,0,0.0,0,0.0,...,0.00,0.0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN
2,2023,Mar,0,0,0,0,0,0.0,0,0.0,...,0.00,0.0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN
3,2023,Apr,0,0,0,0,0,0.0,0,0.0,...,0.00,0.0,0.00,0.00,0.00,NaN,Elec Tariff,EV Tariff,Profit,Gas Tariff
4,2023,May,7,10,26,55,-73,0.0,-18,-18.0,...,4.79,0.0,-7.12,7.24,7.24,NaN,0.390182,0,0,1.367143


In [4]:
# Investigate the unnamed columns
print("Unnamed columns content:")
print(df[["Unnamed: 22", "Unnamed: 23", "Unnamed: 24", "Unnamed: 25"]].dropna().head(10))

print(f"\nYear column type: {df['Year'].dtype}")
print(f"Year unique values: {df['Year'].unique()}")

print(f"\nRows with all zero energy values:")
zero_rows = df[df["Solar Production"] == 0]
print(f"Count: {len(zero_rows)}")
print(zero_rows[["Year", "Month", "Solar Production", "Gas Use", "Elec Use"]])

Unnamed columns content:
    Unnamed: 22 Unnamed: 23 Unnamed: 24 Unnamed: 25
3   Elec Tariff   EV Tariff      Profit  Gas Tariff
4      0.390182           0           0    1.367143
5      0.389242           0           0    1.366538
6      0.388779           0           0    1.366389
7      0.389701           0           0    1.366552
8      0.391538    0.249131    5.733338    1.366667
9      0.387674    0.345934     4.73717    1.366508
10     0.388465    0.357833    4.472056    1.366484
11     0.387119    0.377245    2.698881    1.366509
12     0.368082    0.371606   -0.769745    1.479291

Year column type: object
Year unique values: [2023 2024 2025 2026 'Total']

Rows with all zero energy values:
Count: 4
   Year Month  Solar Production  Gas Use  Elec Use
0  2023   Jan                 0        0         0
1  2023   Feb                 0        0         0
2  2023   Mar                 0        0         0
3  2023   Apr                 0        0         0


In [5]:
print(f"Shape: {df.shape}")

Shape: (38, 26)


In [7]:
print(f"Shape: {zero_rows.shape}")

Shape: (4, 26)


In [14]:
# ------ DATA CLEANING -----------------------------------------------

# Keep the raw data untouched
raw_df = df.copy()

# Work on a clean copy

# Step 1 - Remove rows where Year is 'Total' or energy data is all zero
df = df[df["Year"] != "Total"]
df = df[df["Solar Production"] != 0]

# Step 2 - Convert Year to Integer
df["Year"] = df["Year"].astype(int)

# Step 3 - Rename unnamed tariff columns
df = df.rename(columns={
    "Unnamed: 22": "Elec Tariff",
    "Unnamed: 23": "EV Tariff",
    "Unnamed: 24": "Profit",
    "Unnamed: 25": "Gas Tariff"
})

# Step 4 - Reset the index so it starts from 0
df = df.reset_index(drop=True)

# Confirm results
print(f"Shape after cleaning: {df.shape}")
print(f"Year range: {df['Year'].min()} to {df['Year'].max()}")
print(f"Months: {len(df)}")
print(f"\nFirst 3 rows:")
df.head(3)

Shape after cleaning: (33, 26)
Year range: 2023 to 2026
Months: 33

First 3 rows:


,Year,Month,Gas Use,Elec Use,Solar Production,Elec Import,Elec Export,EV Charger,Net Use,Net Use (-EV),...,Fixed Costs,EV Charger.1,Net House Elec Cost,Net House Cost (Gas & Elec),Overall Cost,Remarks,Elec Tariff,EV Tariff,EV Tariff,Gas Tariff
0,2023,May,7,10,26,55,-73,0.0,-18,-18.0,...,4.79,0.0,-7.12,7.24,7.24,NaN,0.390182,0,0,1.367143
1,2023,Jun,26,186,192,132,-136,0.0,-4,-4.0,...,9.59,0.0,-2.26,42.86,42.86,NaN,0.389242,0,0,1.366538
2,2023,Jul,36,241,162,172,-97,0.0,75,75.0,...,9.91,0.0,28.60,87.70,87.70,NaN,0.388779,0,0,1.366389


In [16]:
# Create a proper datetime column from Year and Month
df["Date"] = pd.to_datetime(df["Year"].astype(str) + " " + df["Month"], format="%Y %b")

# Set it as the index - makes time series operations much easier
df = df.set_index("Date")

print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nFirst 3 rows with new date index:")
df.head(3)

Date range: 2023-05-01 00:00:00 to 2026-01-01 00:00:00

First 3 rows with new date index:


,Year,Month,Gas Use,Elec Use,Solar Production,Elec Import,Elec Export,EV Charger,Net Use,Net Use (-EV),...,Fixed Costs,EV Charger.1,Net House Elec Cost,Net House Cost (Gas & Elec),Overall Cost,Remarks,Elec Tariff,EV Tariff,EV Tariff,Gas Tariff
Date,,,,,,,,,,,,,,,,,,,,,
2023-05-01,2023,May,7,10,26,55,-73,0.0,-18,-18.0,...,4.79,0.0,-7.12,7.24,7.24,NaN,0.390182,0,0,1.367143
2023-06-01,2023,Jun,26,186,192,132,-136,0.0,-4,-4.0,...,9.59,0.0,-2.26,42.86,42.86,NaN,0.389242,0,0,1.366538
2023-07-01,2023,Jul,36,241,162,172,-97,0.0,75,75.0,...,9.91,0.0,28.60,87.70,87.70,NaN,0.388779,0,0,1.366389


In [17]:
# Drop redundant columns now that we have a proper date index
df = df.drop(columns=["Year", "Month"])

# Also check what our data looks like now
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Shape: (33, 24)
Columns: ['Gas Use', 'Elec Use', 'Solar Production', 'Elec Import', 'Elec Export', 'EV Charger', 'Net Use', 'Net Use (-EV)', 'Heating Degree Days', 'Gas Cost', 'Elec Import Cost', 'Export Income', 'Feedback Costs', 'Net Elec Cost', 'Fixed Costs', 'EV Charger.1', 'Net House Elec Cost', 'Net House Cost (Gas & Elec)', 'Overall Cost', 'Remarks', 'Elec Tariff', 'EV Tariff', 'EV Tariff', 'Gas Tariff']


,Gas Use,Elec Use,Solar Production,Elec Import,Elec Export,EV Charger,Net Use,Net Use (-EV),Heating Degree Days,Gas Cost,...,Fixed Costs,EV Charger.1,Net House Elec Cost,Net House Cost (Gas & Elec),Overall Cost,Remarks,Elec Tariff,EV Tariff,EV Tariff,Gas Tariff
Date,,,,,,,,,,,,,,,,,,,,,
2023-05-01,7,10,26,55,-73,0.0,-18,-18.0,132.67,9.57,...,4.79,0.0,-7.12,7.24,7.24,NaN,0.390182,0,0,1.367143
2023-06-01,26,186,192,132,-136,0.0,-4,-4.0,16.93,35.53,...,9.59,0.0,-2.26,42.86,42.86,NaN,0.389242,0,0,1.366538
2023-07-01,36,241,162,172,-97,0.0,75,75.0,5.49,49.19,...,9.91,0.0,28.60,87.70,87.70,NaN,0.388779,0,0,1.366389
